# One agent or four? Topologies and handoffs



> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 17 §17.1–§17.3 · type: concept-lab*



**The promise:** by the end you can decide *whether* to split one agent into a team, and *which* topology fits — by quantifying the costs a slick demo hides.

## 🧠 Why this matters



A multi-agent system *is* a distributed system. Once one agent works, the temptation to add a researcher, a writer, a critic, and a manager arrives on schedule — it demos beautifully and flatters the intuition that intelligence scales like an org chart.



But every hop adds a full model call of latency and cost, errors **compound** across handoffs, and shared state introduces races. Most teams reach for a team too soon. This lab makes the counter-forces *tangible* so you choose a topology on purpose — not because four boxes look more capable than one.

## Objectives & prereqs



**By the end you can:**

- Justify every agent with a *named* force (context isolation, specialization, parallelism, privilege separation) — or admit there isn't one.

- Compute how reliability compounds across handoffs and pick a topology accordingly.

- Write a `Handoff` that doesn't silently drop the one fact that mattered.



**Prereqs:** Ch 16 (reasoning patterns, `RunBudget`); Ch 8 (lost-in-the-middle / context). No API key and no network needed — this lab is offline by design.

In [ ]:
# Setup: stdlib + pydantic only. No API key required.

import os

import random

from dataclasses import dataclass, field



from dotenv import load_dotenv

from pydantic import BaseModel, ValidationError



load_dotenv()  # picks up a .env if you have one; nothing here needs it



# MOCK=1 (default) keeps everything offline and deterministic. This concept-lab

# never makes a live call even with MOCK=0 — the lesson is structural, not API-bound.

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"



SEED = 17

random.seed(SEED)  # any stochastic demo below is reproducible



print(f"MOCK={MOCK}  (offline concept-lab; no keys, no spend)")

## 🧠 Mental model: an agent is an employee



Designing a multi-agent system is **organization design**. An agent is an employee defined by three things:



| Employee | Agent |

|---|---|

| Job description | system prompt |

| Skills | tools |

| Need-to-know access | context + permissions |



You hire a second person only when one person *genuinely cannot do the job* — too much to hold in their head, conflicting roles, or work that must happen in parallel. And like real orgs, every hire adds communication overhead that grows faster than headcount.

In [ ]:
# Encode the 'employee' framing as data so we can reason about a candidate team.

@dataclass

class AgentSpec:

    role: str

    job_description: str   # system prompt, in one line

    skills: list           # tools

    need_to_know: str      # the slice of context this role actually needs



candidate_team = [

    AgentSpec("researcher", "Find and cite facts from company docs",

              ["search_docs"], "the question + the doc base"),

    AgentSpec("writer", "Turn notes into a polished brief",

              [], "the research notes only"),

]



for a in candidate_team:

    tools = str(a.skills or "[]")

    print(f"{a.role:11} | tools={tools:<14} | sees: {a.need_to_know}")

## The four legitimate forces (ranked by how often they actually apply)



1. **Context isolation** — *the strongest.* One window holding research notes, style rules, review criteria, and forty tool results degrades (the lost-in-the-middle problem, Ch 8). Split roles → each agent gets a small, relevant context.

2. **Specialization** — different roles want different prompts, tools, even models (cheap model for triage, strong model for synthesis). Tune and eval each in isolation.

3. **Parallelism** — independent subtasks (research five competitors) fan out and cut wall-clock time.

4. **Privilege separation** — the agent reading untrusted web content must not hold the DB-write tools. Boundaries between agents are security boundaries.



If you can't name one of these for a proposed agent, you don't have a reason to add it.

In [ ]:
# A blunt gate: every agent must claim a named force, or it doesn't get hired.

LEGIT_FORCES = {"context_isolation", "specialization", "parallelism", "privilege_separation"}



def justify(role: str, force: str) -> str:

    if force not in LEGIT_FORCES:

        return f"❌ {role}: '{force}' is not a real force — fold this back into one agent."

    return f"✅ {role}: justified by {force}."



print(justify("researcher", "context_isolation"))

print(justify("writer", "context_isolation"))

print(justify("critic", "it_seems_more_capable"))  # the non-force everyone reaches for

## 🔮 Predict: error compounds across handoffs



Suppose a pipeline has **three stages in series**, each 90% reliable on its own (it produces a correct, complete handoff 9 times out of 10).



**Predict before you run the next cell:** what is the end-to-end success rate? Is it closer to 90%, 80%, or 70%?

In [ ]:
# Each stage must succeed for the whole chain to succeed -> probabilities multiply.

stage_reliability = 0.90

n_stages = 3



analytic = stage_reliability ** n_stages

print(f"Analytic end-to-end success: {stage_reliability}^{n_stages} = {analytic:.2f}")



# Monte-Carlo the same chain to *watch* error compound (seeded -> reproducible).

TRIALS = 20_000

successes = 0

for _ in range(TRIALS):

    if all(random.random() < stage_reliability for _ in range(n_stages)):

        successes += 1

empirical = successes / TRIALS

print(f"Empirical over {TRIALS:,} runs:    {empirical:.2f}")

**What you just saw.** Three 90%-reliable stages don't give you 90% — they give you 0.9³ = **0.73**. Add a fourth and you're at 0.66. This is why a pipeline of agents can feel *less* reliable than the single agent it replaced: you multiplied the failure surface. More agents is more places for a handoff to go wrong.

## The four working topologies



Four named shapes cover nearly everything in production. Below is a tiny **mock** of each — no LLM calls, just enough structure to feel the shape and its watch-out.



| Topology | Fit | Watch out for |

|---|---|---|

| Supervisor / worker | One coordinator decomposes, delegates, integrates. The production default. | Supervisor becomes bottleneck / single point of failure |

| Pipeline | Fixed stages, each transforms the last. Deterministic, per-stage evals. | Errors compound; rigid when the task doesn't fit the stages |

| Debate | Agents argue opposing positions; a judge decides. Better calibration. | 2–3× cost for marginal gain on routine tasks |

| Blackboard | Agents read/write a shared workspace, react opportunistically. | Hardest to bound/debug; races on shared state |

In [ ]:
# Mock 'agents' are just functions string -> string. The topology is the wiring.

def upper(x):    return x.upper()

def exclaim(x):  return x + "!"

def shorten(x):  return x[:24]



def supervisor_worker(goal, workers):

    # One coordinator routes the goal to a chosen worker and owns the result.

    chosen = workers[0]               # a real supervisor would *decide*; we fix it for the demo

    return f"[supervisor] integrated -> {chosen(goal)}"



def pipeline(goal, stages):

    x = goal

    for stage in stages:              # each stage transforms the previous output

        x = stage(x)

    return x



def debate(claim, judge):

    pro, con = f"PRO: {claim}", f"CON: not {claim}"

    return judge(pro, con)            # a judge resolves; costs 2+ agent calls



def blackboard(goal, agents, board):

    for agent in agents:              # everyone reacts to the shared board, in order here

        board.append(agent(goal))     # ...but in reality concurrently -> races

    return board



print("supervisor:", supervisor_worker("summarize q3", [upper]))

print("pipeline:  ", pipeline("summarize q3", [upper, exclaim, shorten]))

print("debate:    ", debate("NRR improved", lambda p, c: f"judge picks: {p}"))

print("blackboard:", blackboard("q3", [upper, exclaim], board=[]))

**The headline:** in practice the **supervisor/worker** topology absorbs the others — a supervisor may run two workers as a debate, or arrange three in a pipeline for one subtask. Start there. The exotic topologies are seasoning, not the meal.

## ⚠️ Pitfall: the failure is a *lost message*, not a dumb agent



Multi-agent failures are overwhelmingly **interface** failures. The researcher found the answer, but the free-text summary handed to the writer dropped the key fact; the supervisor asked for X and the worker solved adjacent-X. Watch a free-text handoff silently lose the number that the whole brief depended on.

In [ ]:
# A realistic free-text handoff: chatty, human-readable... and lossy.

free_text_handoff = (

    "Hey, looked into Q3. Retention is down and mid-market churn is the story. "

    "The analytics add-on came up a lot in exit surveys. Can you write it up?"

)



# The one fact the brief actually needs: NRR fell to 104% from 119%.

KEY_FACT = "NRR 104% (down from 119%)"

print("Key fact survived the handoff?", KEY_FACT in free_text_handoff)  # -> False

In [ ]:
# The fix (Ch 15, Ch 17.3): a *structured* Handoff, validated at the boundary.

class Handoff(BaseModel):

    task: str                    # what the receiving agent must do

    context: str                 # everything needed; assume NO shared memory

    artifacts: list[str] = []    # ids/paths of produced outputs

    open_questions: list[str] = []  # known unknowns, so they aren't re-derived

    done_criteria: str           # how the receiver knows it's finished



good = Handoff(

    task="Write a 1-paragraph brief on the Q3 retention decline.",

    context="NRR fell to 104% from 119%, driven by mid-market churn after the "

            "March pricing change. 61% of churned accounts cited unclear ROI on "

            "the analytics add-on.",

    artifacts=["doc-001", "doc-002"],

    open_questions=["Is enterprise NRR holding?"],

    done_criteria="One paragraph, every claim traceable to a source id.",

)

print("Key fact survived?", "104%" in good.context and "119%" in good.context)  # -> True

In [ ]:
# Validation at the boundary: a bad handoff fails LOUDLY here, not 3 agents later.

def validate_handoff(raw: dict) -> Handoff:

    try:

        return Handoff(**raw)

    except ValidationError as e:

        # Surface the missing field where it bites, not as a mystery downstream.

        missing = [err["loc"][0] for err in e.errors() if err["type"] == "missing"]

        raise ValueError(f"Rejected handoff — missing required field(s): {missing}") from None



try:

    validate_handoff({"task": "write it up", "context": "retention stuff"})  # no done_criteria

except ValueError as e:

    print(e)

**What you just saw.** Structure buys three wins at once: nothing important is dropped, every hop is *validatable* (the bad handoff failed at the boundary, not quietly three agents later), and your traces become readable documents instead of prompt soup.

## 🎯 Senior lens



The senior move isn't picking the cleverest topology — it's re-asking, at every design review, **"would one agent with better tools and context do this more simply?"** The default for the whole book holds here: the simplest design that meets the requirements. You split an agent the way you'd split a monolith — only when a *named* force demands it, knowing each split multiplies your failure surface and your debugging cost. A four-agent system that's really a one-agent system with extra latency is the most common (and most expensive) mistake in this chapter.

## Recap



- A multi-agent system **is** a distributed system; add an agent only for a named force.

- The forces, ranked: **context isolation** (strongest), specialization, parallelism, privilege separation.

- Reliability **multiplies**: three 90% stages → 0.73. More agents = more failure surface.

- **Supervisor/worker** is the default and absorbs the other topologies as sub-patterns.

- The dominant failure mode is a **lost message** — fix it with a structured, boundary-validated `Handoff`, not better in-agent prompts.

## Exercises



Each task *changes* something and asks you to predict the result first.



1. **Compounding, harder.** Re-run the reliability cell with `n_stages = 5` and `stage_reliability = 0.95`. 🔮 Predict the end-to-end rate before you run it. Is a longer chain of *more* reliable stages safe?

2. **A third specialist.** Add a `critic` agent to `candidate_team`. Name the force that justifies it — or argue it should be folded into the writer. Make `justify()` agree.

3. **Break a handoff on purpose.** Construct a `Handoff` that *passes* validation but still loses a fact (a complete-looking `context` that omits the number). What does this tell you about the limits of schema validation — and what eval (Ch 22) would catch it?

4. **Pick a topology.** For “research 5 competitors and write one comparison brief,” which topology fits, and which force(s) justify the split? Write your answer as a comment.

In [ ]:
# Exercise 1: reliability with n_stages=5, stage_reliability=0.95


In [ ]:
# Exercise 2: add a 'critic' to candidate_team and justify (or reject) it


In [ ]:
# Exercise 3: a Handoff that validates but still loses a fact


In [ ]:
# Exercise 4: choose a topology + name the force(s)


## Next



You decided *whether* and *which*. Next, **build** it: a supervisor that owns the goal and budget, a researcher wired to RAG, and a writer with no retrieval — each specialist exposed to the supervisor *as a tool*.



- ➡️ Next notebook: [`17-02-supervisor-and-specialists.ipynb`](./17-02-supervisor-and-specialists.ipynb) (the chapter's 🔧 Build, §17.5)

- 📘 Book: §17.1–§17.3

- 🔧 Blueprint (the production version): [`../../../blueprints/multi-agent-supervisor/`](../../../blueprints/multi-agent-supervisor/)

- 🏁 Capstone: feeds [`../../../capstone/agents/`](../../../capstone/agents/) — checkpoint `ch17-supervisor`.